# 🤖 Weblume Coder — موديل بيعمل ويعدّل مواقع بأسلوبك

النوت بوك ده بيدرّب موديل برمجة **بتاعك إنت** على:
- **مواقع كاملة بأسلوب Weblume**: داكن وفخم، عناوين كبيرة بلون مميز، أنيميشن ناعم، وواتساب.
- **تعديل المواقع**: ألوان، خطوط، سكاشن، لغتين، وضع ليلي…
- **سكاشن لوحدها** وأسئلة البرمجة والنشر، **والرد على العملاء**.

بيانات التدريب **جاهزة جوه النوت بوك** (أكتر من ٢٧٠ مثال، ومعاهم موقع Weblume بتاعك). **مش محتاج أي مفتاح.**

مظبوط على جهازك: **RTX 4060 (8 جيجا)** ← موديل **Qwen2.5-Coder-7B** هيشتغل كله على كارت الشاشة.

## الخطوات
1. **Runtime ← Change runtime type ← T4 GPU ← Save**.
2. **Runtime ← Run all**.
3. وافق على إذن **Google Drive** (عشان التدريب يتحفظ أول بأول).
4. استنى حوالي ساعة ونص. لو Colab فصل: **Run all** تاني ← هيكمّل من آخر نقطة.
5. في الآخر هتلاقي في Drive فولدر **weblume_model** فيه `weblume-lora.gguf` (≈ ٨٠ ميجا) و `Modelfile`.

## ⚙️ الإعدادات

In [ ]:
#@title الإعدادات
# Coder-7B ← الأنسب لجهازك (RTX 4060 8GB) — حجمه ≈ 4.7 جيجا
# Coder-3B ← تدريب أسرع بالنص بس أضعف في التصميم
BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct"  #@param ["unsloth/Qwen2.5-Coder-7B-Instruct", "unsloth/Qwen2.5-Coder-3B-Instruct"]
EPOCHS = 1     #@param {type:"integer"}
# لو فيه تدريب قديم في Drive يكمّل عليه (أسرع بكتير) ويتدرّب على نص البيانات بس
CONTINUE_TRAINING = True  #@param {type:"boolean"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
MAX_LEN = 16384

SYSTEM_PROMPT = (
    "اسمك Weblume AI، مساعد شركة Weblume لعمل المواقع. انت مبرمج Full-Stack محترف ومصمم واجهات مبدع، "
    "بتتكلم مصري بسيط وودود، والمستخدم ممكن يكون مبتدئ.\n"
    "- لو ماحددش لغة برمجة: اكتب موقع HTML واحد كامل فيه CSS وJS، شغال على الموبايل، من غير مكتبات لازمة.\n"
    "- التصميم لازم يبقى مبتكر واحترافي: هوية بصرية خاصة بالمشروع، خطوط Google Fonts مميزة، "
    "مسافات مريحة، أنيميشن ناعم، وتفاصيل صغيرة تفرق. ممنوع التصميم العادي المكرر.\n"
    "- النصوص حقيقية ومقنعة بلغة الموقع، ومن غير Lorem ipsum. العربي RTL.\n"
    "- في التعديل: نفّذ المطلوب بالظبط من غير ما تبوّظ باقي الموقع، ورجّع الملف كامل بعد التعديل.\n"
    "- اكتب الكود جوه ```html وبعده نقط قصيرة باللي عملته.\n"
    "- الصور: <img src=\"https://placehold.co/WxH\" data-photo=\"english search words\" alt=\"وصف عربي\" width=\"W\" height=\"H\">، وماتألفش لينكات صور.\n"
    "- ماتكتبش أرقام تليفونات من عندك؛ لو مش متاحة حط مكانها واضح للتعديل."
)
print("✅", BASE_MODEL, "| Epochs:", EPOCHS)

## 1️⃣ كارت الشاشة + الأدوات + Google Drive

In [ ]:
import subprocess, os
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("❌ مفيش كارت شاشة! Runtime ← Change runtime type ← T4 GPU ← Save، وبعدين Run all.")
print("✅ كارت الشاشة:", gpu.stdout.strip())

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/weblume_model"
else:
    WORK = "/content/weblume_model"
os.makedirs(WORK, exist_ok=True)
DATA_FILE = f"{WORK}/train_data.jsonl"
print("📁 الشغل بيتحفظ في:", WORK)

In [ ]:
%%capture
!pip install -q unsloth

## 2️⃣ بيانات التدريب
اتكتبت بأسلوب Weblume: ١٤٠ موقع لـ٢٤ نوع نشاط (كافيهات ومطاعم وعيادات وجيمات وبراندات وشركات وبورتفوليو وتطبيقات)، و١٢ ستايل ألوان، و٤ أشكال هيرو، و٧٠ تعديل، و٤٠ سكشن لوحده، وأسئلة برمجة، وردود عملاء، **وموقع Weblume بتاعك**.

In [ ]:
#@title جهّز بيانات التدريب (مكتوبة بأسلوب Weblume)
import base64, io, zipfile, sys, json, random, importlib
GEN_ZIP = """
...
"""
zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(GEN_ZIP.split())))).extractall("/content/weblume_gen")
sys.path.insert(0, "/content/weblume_gen")
import dataset; importlib.reload(dataset)
DATA = dataset.make_dataset() + dataset.mine_examples("/content/weblume_gen/mine")
from collections import Counter
print("✅ بيانات التدريب:", len(DATA), dict(Counter(d["kind"] for d in DATA)))

In [ ]:
#@title مواقع Phainora المعتمدة (24 مواقع كاملة بأنيميشن وعربي/إنجليزي)
# كل موقع بيتكرر مرتين علشان الموديل ياخد المستوى ده. الصور فيها صيغة ثابتة يتبدل مكانها صور حقيقية بعدين.
import base64, io, zipfile, json, os
PHAINORA_ZIP = """
...
AA==
"""
os.makedirs("/content/phainora", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode("".join(PHAINORA_ZIP.split())))).extractall("/content/phainora")
for ex in json.load(open("/content/phainora/examples.json", encoding="utf-8")):
    html = open("/content/phainora/" + ex["file"], encoding="utf-8").read().strip()
    ans = f"تمام! ده الموقع كامل في ملف واحد 👇\n\n```html\n{html}\n```\n\n- الصور مكتوبة بالصيغة الثابتة (data-photo)، وسكريبت الصور بيحط مكانها صور حقيقية.\n- زرار EN بيقلب الموقع إنجليزي.\nلو حابب أغيّر أي حاجة قولّي."
    DATA += [{"kind": "mine", "messages": [{"role": "user", "content": ex["request"]}, {"role": "assistant", "content": ans}]}] * 2
print("✅ اتضاف", len(json.load(open("/content/phainora/examples.json", encoding="utf-8"))) * 2, "مثال من مواقع Phainora — الإجمالي", len(DATA))


### مواقعك التانية (اختياري)
لو عندك مواقع تانية عملتها (ملف HTML واحد)، فعّل الاختيار وارفعها، والموديل هيتعلم منها أسلوبك أكتر.

In [ ]:
#@title ارفع مواقع زيادة
UPLOAD_MY_SITES = False  #@param {type:"boolean"}
if UPLOAD_MY_SITES:
    from google.colab import files
    import os, re
    extra = []
    for name, raw in files.upload().items():
        html = raw.decode("utf-8", "ignore")
        if "<html" not in html.lower(): print("⚠️ اتجاهل:", name); continue
        html = re.sub(r"data:([a-z/+]+);base64,[A-Za-z0-9+/=]{100,}", "assets/image.png", html)
        open(f"/content/weblume_gen/mine/{name}", "w", encoding="utf-8").write(html)
        title = (re.findall(r"<title>(.*?)</title>", html, re.S) or [name])[0].strip()
        extra.append((name, f"اعملي موقع لـ{title} بأسلوب Weblume"))
    DATA += dataset.mine_examples("/content/weblume_gen/mine", extra)[len(dataset.MINE) * 3:]
    print("✅ بقى عندك", len(DATA), "مثال")

In [ ]:
#@title شوف عينة من المواقع اللي هيتعلم منها
import re, html as _h
from IPython.display import HTML, display
def html_of(text):
    m = re.search(r"```html\s*(.*?)```", text, re.S | re.I)
    return m.group(1).strip() if m else ""
s = random.choice([d for d in DATA if d["kind"] == "site"])
print("🧑", s["messages"][0]["content"])
display(HTML(f'<iframe srcdoc="{_h.escape(html_of(s["messages"][1]["content"]))}" style="width:100%;height:560px;border:1px solid #333;border-radius:8px"></iframe>'))

## 4️⃣ التدريب (LoRA على كارت T4 المجاني)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch, glob, random
from datasets import Dataset

ck_num = lambda p: int(p.rsplit("-", 1)[-1])
OLD = sorted(glob.glob(f"{WORK}/checkpoints/checkpoint-*"), key=ck_num)
CK_DIR = f"{WORK}/checkpoints_v2" if OLD else f"{WORK}/checkpoints"

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

TRAIN = DATA
if OLD and CONTINUE_TRAINING:
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    res = set_peft_model_state_dict(model, load_file(f"{OLD[-1]}/adapter_model.safetensors"))
    print("↩️ بكمّل على التدريب القديم:", OLD[-1], "| مفاتيح ناقصة:", len([k for k in getattr(res, "missing_keys", []) if "lora_" in k]))
    rnd = random.Random(7)
    TRAIN = [d for d in DATA if d["kind"] in ("chat", "mine") or rnd.random() < 0.5]

def fmt(ex):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}] + ex["messages"]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
texts = [fmt(e) for e in TRAIN]
keep = [t for t in texts if len(tokenizer(t, add_special_tokens=False)["input_ids"]) <= MAX_LEN]
print(f"✅ {len(keep)} مثال جاهز للتدريب (اتشال {len(texts) - len(keep)} عشان أطول من {MAX_LEN} توكن)")
ds = Dataset.from_list([{"text": t} for t in keep])

In [ ]:
from trl import SFTTrainer, SFTConfig
cfg = dict(dataset_text_field="text", max_seq_length=MAX_LEN, max_length=MAX_LEN, per_device_train_batch_size=1,
           gradient_accumulation_steps=8, num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=3,
           logging_steps=5, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
           seed=3407, output_dir=CK_DIR, report_to="none", save_strategy="steps", save_steps=8, save_total_limit=1,
           fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported())
try:
    args = SFTConfig(**cfg)
except TypeError:  # نسخ trl بتقبل اسم واحد بس من الاتنين
    try:
        cfg.pop("max_seq_length"); args = SFTConfig(**cfg)
    except TypeError:
        cfg["max_seq_length"] = cfg.pop("max_length"); args = SFTConfig(**cfg)
try:
    trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=args)
except TypeError:
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=args)
try:  # يتعلم من الردود بس، مش من الأسئلة
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
except Exception as e:
    print("ℹ️ هندرّب على النص كله:", e)

import glob as _g
resume = bool(_g.glob(f"{CK_DIR}/checkpoint-*"))
if resume: print("↩️ بكمّل التدريب من آخر نقطة اتحفظت")
stats = trainer.train(resume_from_checkpoint=resume)
print(f"✅ خلص التدريب في {stats.metrics['train_runtime']/60:.1f} دقيقة — loss: {stats.metrics['train_loss']:.3f}")

## 5️⃣ جرّب الموديل بنفسك

In [ ]:
FastLanguageModel.for_inference(model)
def ask(q, max_new_tokens=7000):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt")
    if not torch.is_tensor(ids): ids = ids["input_ids"]
    ids = ids.to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.3, top_p=0.9, do_sample=True, repetition_penalty=1.05)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def show(answer):
    h = html_of(answer)
    print(re.sub(r"```html.*?```", "[… الكود …]", answer, flags=re.S)[:1500])
    if h:
        display(HTML(f'<iframe srcdoc="{_h.escape(h)}" style="width:100%;height:560px;border:1px solid #333;border-radius:8px"></iframe>'))
        open("test_site.html", "w", encoding="utf-8").write(h)

print("⏳ بيكتب موقع كامل… (حوالي ٥ دقايق على كارت Colab، وعلى جهازك أسرع)")
show(ask("اعملي موقع لمحل كنافة وحلويات شرقية اسمه Sweet Nile في المنصورة، ستايل داكن فخم بلمسات دهبي وفيه منيو وطلب واتساب"))

In [ ]:
#@title اطلب أي موقع
طلبك = "اعملي لاندنج بيدج لجيم اسمه Iron House بالإنجليزي، ستايل نيون"  #@param {type:"string"}
show(ask(طلبك))

## 6️⃣ تصدير الموديل لـ Ollama (ملف صغير ≈ ٨٠ ميجا)
بدل ملف ٥ جيجا، بنصدّر **الإضافة** اللي اتعلمها الموديل بس، وOllama بيركّبها على `qwen2.5-coder` الرسمي على جهازك.

In [ ]:
import glob, os, subprocess
ck_num = lambda p: int(p.rsplit("-", 1)[-1])
BEST = sorted(glob.glob(f"{CK_DIR}/checkpoint-*"), key=ck_num)[-1]
print("📦 هصدّر:", BEST)
BASE_HF = {"unsloth/Qwen2.5-Coder-7B-Instruct": "Qwen/Qwen2.5-Coder-7B-Instruct", "unsloth/Qwen2.5-Coder-3B-Instruct": "Qwen/Qwen2.5-Coder-3B-Instruct"}[BASE_MODEL]
OLLAMA_BASE = {"unsloth/Qwen2.5-Coder-7B-Instruct": "qwen2.5-coder:7b", "unsloth/Qwen2.5-Coder-3B-Instruct": "qwen2.5-coder:3b"}[BASE_MODEL]
if not os.path.exists("/content/llama.cpp/convert_lora_to_gguf.py"):
    subprocess.run("git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp", shell=True, check=True)
    subprocess.run("pip install -q -e /content/llama.cpp/gguf-py", shell=True, check=True)
r = subprocess.run(["python", "/content/llama.cpp/convert_lora_to_gguf.py", BEST, "--base-model-id", BASE_HF,
                    "--outtype", "f16", "--outfile", f"{WORK}/weblume-lora.gguf"], capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
if not os.path.exists(f"{WORK}/weblume-lora.gguf"): raise SystemExit("❌ التصدير فشل — ابعت صورة للرسالة اللي فوق")

modelfile = (f"FROM {OLLAMA_BASE}\nADAPTER ./weblume-lora.gguf\n\nSYSTEM \"\"\"" + SYSTEM_PROMPT.replace('"""', "'") + "\"\"\"\n\n"
             "PARAMETER temperature 0.3\nPARAMETER top_p 0.9\nPARAMETER repeat_penalty 1.05\nPARAMETER num_ctx 16384\nPARAMETER num_predict 12000\n")
open(f"{WORK}/Modelfile", "w", encoding="utf-8").write(modelfile)
print("✅ جاهز:", round(os.path.getsize(f"{WORK}/weblume-lora.gguf") / 1e6), "ميجا ← Drive/weblume_model (weblume-lora.gguf + Modelfile)")

In [ ]:
#@title تنزيل مباشر (لو مش عايز تنزّل من Drive)
DOWNLOAD_NOW = False  #@param {type:"boolean"}
if DOWNLOAD_NOW:
    from google.colab import files
    files.download(f"{WORK}/Modelfile"); files.download(f"{WORK}/weblume-lora.gguf")
else:
    print("📁 نزّل weblume-lora.gguf و Modelfile من Google Drive ← فولدر weblume_model")

## ✅ خلصنا!
1. من Google Drive ← فولدر **weblume_model** نزّل `weblume-lora.gguf` و `Modelfile`.
2. حطهم جنب `install_model.bat` ودوس عليه دبل كليك (هينزّل `qwen2.5-coder:7b` الرسمي أول مرة ≈ ٤.٧ جيجا).
3. Weblume AI ← الإعدادات ← **💻 جهازي** ← الموديل: **weblume**.

عايز تحسّنه؟ ارفع مواقع أكتر من شغلك في خطوة «ارفع مواقع زيادة» وشغّل Run all تاني — هيكمّل على التدريب اللي فات.